In [2]:
import numpy as np
import ase
from ase import Atoms
from ase import io
from ase.io.espresso import write_espresso_in as write_in
from ase.io.espresso import read_espresso_out as read_out
from ase.io.lammpsdata import write_lammps_data
from ase.io.lammpsdata import read_lammps_data
from ase.io.lammpsrun import read_lammps_dump_text
from ase.io import write
from ase.io import read
from ase.calculators.espresso import Espresso
from ase.build import surface
from ase.build import sort
from ase.constraints import FixAtoms
import os
import sys
import matplotlib
matplotlib.use("agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pylab import *
from matplotlib import rc

In [7]:

################################################################# Pt bulk QE.sub File creation ##################################################

main_path=f"./../"
cif_path = input(f"Enter the CIF path (e.g., systems/Si/): ")
output_dir = os.path.join(main_path, "bulk_optimization")
#user_email = "ch24d003@smail.iitm.ac.in"
output_file_path = os.path.join(output_dir, "qe.sub")

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the content of the SLURM submission script as a multi-line string
# The f-string is used to insert the user's email dynamically
script_content = f"""#!/bin/bash
#SBATCH --job-name=scf_Pt_Bulk_lattice
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=24
#SBATCH --partition=standard
#SBATCH --time=04:00:00
#SBATCH --mail-type=end

mpirun -np 48 -mca coll_hcoll_enable 0 $HOME/qe-6.7/bin/pw.x -npool 1 -in pw.in > pw.out
"""

# Write the content to the file
try:
    with open(output_file_path, "w") as f:
        f.write(script_content)
    print(f"Successfully created: {output_file_path}")
except IOError as e:
    print(f"Error: Could not write to file {output_file_path}. {e}")
    sys.exit(1)


################################################################# Pt pw.in File creation ##################################################
pseudo_dir = f"./../pseudo/"
pseudopotentials = {'Pt': 'Pt_ONCV_PBE-1.2.upf'}
atoms = io.read(cif_path +"Pt.cif",format='cif')
ntyp = len(set(atoms.get_chemical_symbols()))
nat = len(atoms)
input_qe = {
    'calculation': 'vc-relax',
    'prefix': 'vc',
    'pseudo_dir': pseudo_dir,
    'outdir': './outdir',
    'restart_mode': 'from_scratch', 
    'nstep': 100,
    'etot_conv_thr': 0.0001,
    'forc_conv_thr': 0.001,
    'max_seconds': 3500,
    'system': {
        'ibrav': 0,
        'ntyp': ntyp,
        'nat': nat,
        'ecutwfc': 70,
        'ecutrho': 280, 
        'nbnd': 44,
        'occupations': 'smearing', 
        'smearing': 'marzari-vanderbilt',
        'degauss': 0.02,
        'input_dft': 'PBE'
    },
    'electrons': {
        'mixing_beta': 0.1,
        'diagonalization': 'david',
        'conv_thr': 1e-06,
        'mixing_ndim': 8,
    },
    'ions': { 
        'ion_dynamics':'bfgs' 
    },
    'cell': { 
        'cell_dynamics':'bfgs' 
    },
}

kpts = (8, 8, 8)                    # K-point mesh size
offset = (0, 0, 0)                  # Offset for the k-point mesh

# --- Generate QE input file for selected configuration ---
pw_in_path = os.path.join(output_dir, 'pw.in')

with open(pw_in_path, 'w') as f:
    write(f, atoms, format='espresso-in', input_data=input_qe, pseudopotentials=pseudopotentials, kpts=kpts, koffset=offset)

print(f"Successfully created: {pw_in_path}")

Enter the CIF path (e.g., systems/Si/):  ../energycutoffandkpointsoptimization/


Successfully created: ./../bulk_optimization/qe.sub
Successfully created: ./../bulk_optimization/pw.in


In [8]:
######################################### Pt Bulk Optimization and Flat Surface Creation ##################################################

# Read the optimized structure from the Quantum ESPRESSO output
atoms = ase.io.read("./../bulk_optimization/pw.out", format='espresso-out')
opticell = atoms.get_cell()

# Read the initial structure from the CIF file
cif_atoms = ase.io.read("../energycutoffandkpointsoptimization/Pt.cif", format='cif')

# Set the cell of the CIF file object to the optimized cell.
# The `scale_atoms=True` argument rescales the atomic positions to maintain their fractional coordinates within the new cell.
cif_atoms.set_cell(opticell, scale_atoms=True)

# Print the cell to confirm the change in memory
print("Updated cell of CIF file:")
print(cif_atoms.get_cell())

# Write the modified Atoms object to a new file.
# The filename and file format ('cif') must be specified.
ase.io.write("./../bulk_optimization/Pt_optimized.cif", cif_atoms, format='cif')

print("\nSuccessfully wrote the new file 'Pt_optimized.cif'")

Updated cell of CIF file:
Cell([3.972454001, 3.972454001, 3.972454001])

Successfully wrote the new file 'Pt_optimized.cif'
